# Experimental Data Simulation, Cleaning, and Reshaping

In [19]:
import numpy as np
import os
import pandas as pd

Simulate the data with NumPy

In [2]:
# Set random seed for reproducibility.
np.random.seed(1)

In [3]:
# Parameters
n_participants = 30
n_trials = 20
conditions = ['Congruent', 'Incongruent']

In [4]:
# Means and standard deviations for each condition
means = {'Congruent': 500, 'Incongruent': 600}
stds = {'Congruent': 50, 'Incongruent': 60}

In [6]:
# Simulate reaction times
data = {}
for cond in conditions:
    # Shape: participants * trials
    data[cond] = np.random.normal(loc=means[cond], scale=stds[cond],
                                  size=(n_participants, n_trials))

In [7]:
# Introduce some missing values (simulate missing trials)
nan_mask = np.random.rand(n_participants, n_trials) < 0.05
for cond in conditions:
    data[cond][nan_mask] = np.nan

In [8]:
# Checkup
print(data['Congruent'].shape)
print(data['Congruent'][:5, :5])

(30, 20)
[[581.21726818 469.41217932 473.59141239 446.35156889 543.27038147]
 [444.96904114 557.23618549 545.07953603 525.12471695 545.04279746]
 [490.40822238 455.6185518  462.64208531 584.62273005 502.54038774]
 [462.28010295 562.64340776 525.64649102 485.09535824 524.42590733]
 [488.88359287 489.96209655 509.32806955 520.50258236 509.91498601]]


Save and load the data

In [12]:
# Define save directory
save_dit = r"F:\DocHob\Education\Programing\Python_Language\PythonProjects\Raw_Data"

# Save each condition as a separate CSV
np.savetxt(os.path.join(save_dit, "Experiment02_MiniprojCongruent_Raw"),
          data['Congruent'],
          delimiter=",",
          fmt="%.5f") # It keeps numeric precision and preserves NaNs.

np.savetxt(os.path.join(save_dit, "Experiment02_MiniprojIncongruent_Raw"),
          data['Incongruent'],
          delimiter=",",
          fmt="%.5f")

In [13]:
# Load raw data back from disk
congruent_raw = np.genfromtxt(
    os.path.join(save_dit, "Experiment02_MiniprojCongruent_Raw"),
    delimiter=","
)

incongruent_raw = np.genfromtxt(
    os.path.join(save_dit, "Experiment02_MiniprojIncongruent_Raw"),
    delimiter=","
)

# Reconstruct data dictionary
data = {
    "Congruent": congruent_raw,
    "Incongruent": incongruent_raw
}

Clean the data

In [15]:
# Define plausible RT range
min_rt = 200
max_rt = 1500

# Cleaning function
def clean_data(rt_array):
    # Mask invalid values
    clean_array = np.where((rt_array >= min_rt) & (rt_array <= max_rt), rt_array, np.nan)
    return clean_array

# Apply cleaning
for cond in conditions:
    data[cond] = clean_data(data[cond])

Compute descriptive statistics

In [18]:
# Compute participant-level stats
participant_means = {}
participant_stds = {}

for cond in conditions:
    participant_means[cond] = np.nanmean(data[cond], axis=1) # Mean across trials
    participant_stds[cond] = np.nanstd(data[cond], axis=1)

# Group-level statistics
group_means = {cond: np.nanmean(participant_means[cond]) for cond in conditions}
group_stds = {cond: np.nanstd(participant_stds[cond]) for cond in conditions}

print("Participant means (first 5):", participant_means['Congruent'][:5])
print("Group-level means:", group_means)

Participant means (first 5): [498.08460222 501.65954944 510.2069485  508.654289   506.44372778]
Group-level means: {'Congruent': np.float64(502.8616250067997), 'Incongruent': np.float64(598.23364124132)}


Prepare arrays for pandas/statistical analysis

In [22]:
# Create long-format dataframe
rows = []
for p in range(n_participants):
    for cond in conditions:
        for t in range(n_trials):
            rows.append([p+1, cond, t+1, data[cond][p, t]])

df = pd.DataFrame(rows, columns=['Participant', 'Condition', 'Trial', 'RT'])

print(df.head(10))

   Participant  Condition  Trial         RT
0            1  Congruent      1  581.21727
1            1  Congruent      2  469.41218
2            1  Congruent      3  473.59141
3            1  Congruent      4  446.35157
4            1  Congruent      5  543.27038
5            1  Congruent      6  384.92307
6            1  Congruent      7  587.24059
7            1  Congruent      8  461.93965
8            1  Congruent      9  515.95195
9            1  Congruent     10  487.53148
